# 07. Демо-кейсы и фикстуры для моков и сидирования БД

ТЗ §2.1.2, 2.1.11–2.1.13, 2.6.2–2.6.3. Результат ноутбука:
- ≥ 60 демо-компаний (15 «сценарных» кейсов с заранее известным исходом + 50 типичных, стратифицированных по рейтингам);
- JSON-фикстуры по источникам (ЕГРЮЛ, финансы, БКИ, ФНС, обороты) для **DevOps-моков** и общий `companies.json` для **сидирования БД Backend** — ИНН везде единые;
- таблица **«ИНН → ожидаемое поведение»** для TeamLead (`docs/test-data.md`);
- пометка «данные синтетические» в каждом файле.

> Колонка «референсное решение» — **иллюстрация** того, как `credit-core` может интерпретировать оценку. Пороги одобрения принадлежат `credit-core` (конфигурация Backend), не модели.

In [1]:
%%writefile ../data/fixtures.py
"""Конвертация «плоская строка заявки» <-> вложенный JSON-фикстура (формат для моков и сидирования БД).
Все данные синтетические."""
from __future__ import annotations

import math

import numpy as np
import pandas as pd

NOTICE = ("СИНТЕТИЧЕСКИЕ ДАННЫЕ: сгенерированы командой хакатона; к реальным лицам и организациям не относятся. "
          "ИНН/ОГРН из несуществующего диапазона (код региона 00).")

COMPANY = [("id", "company_id"), ("inn", "inn"), ("ogrn", "ogrn"), ("name", "company_name"), ("legalForm", "legal_form"), ("regDate", "reg_date"),
           ("okved", "okved"), ("industry", "industry"), ("regionCode", "region_code"), ("regionName", "region_name"), ("regionTier", "region_tier"),
           ("employees", "employees"), ("taxRegime", "tax_regime"), ("status", "status")]
FIN = [("revenue", "revenue"), ("costOfSales", "cost_of_sales"), ("ebitda", "ebitda"), ("netProfit", "net_profit"), ("totalAssets", "total_assets"),
       ("totalLiabilities", "total_liabilities"), ("equity", "equity"), ("shortTermLiabilities", "short_term_liabilities"),
       ("currentAssets", "current_assets"), ("receivables", "receivables"), ("payables", "payables")]
TURN = [("creditTurnover", "turnover_credit_m"), ("debitTurnover", "turnover_debit_m"), ("endBalance", "balance_end_m")]
TURN_SCALAR = [("bankTenureMonths", "bank_tenure_months"), ("counterpartiesCnt", "counterparties_cnt")]
BKI = [("hasHistory", "bki_has_history"), ("historyMonths", "bki_history_months"), ("openLoans", "bki_open_loans"), ("closedLoans", "bki_closed_loans"),
       ("openAmount", "bki_open_amount"), ("outstandingDebt", "bki_outstanding_debt"), ("monthlyPayment", "bki_monthly_payment"),
       ("closedAmount", "bki_closed_amount"), ("overdue30", "bki_overdue_30_cnt"), ("overdue60", "bki_overdue_60_cnt"), ("overdue90", "bki_overdue_90_cnt"),
       ("maxOverdueDays", "bki_max_overdue_days"), ("currentOverdueAmount", "bki_current_overdue_amount"), ("defaultsCnt", "bki_defaults_cnt"),
       ("inquiries3m", "bki_inquiries_3m"), ("inquiries12m", "bki_inquiries_12m")]
FNS = [("taxDebt", "fns_tax_debt"), ("accountBlocked", "fns_account_blocked"), ("massAddress", "fns_mass_address"), ("massDirector", "fns_mass_director"),
       ("disqualified", "fns_disqualified"), ("bankruptcy", "fns_bankruptcy")]
APP = [("applicationDate", "application_date"), ("productCode", "product_code"), ("requestedAmount", "requested_amount"), ("termMonths", "term_months"),
       ("indicativeRate", "indicative_rate"), ("purpose", "purpose"), ("ownContributionPct", "own_contribution_pct"),
       ("collateralType", "collateral_type"), ("collateralValue", "collateral_value")]
_INT_COLS = {"employees", "region_tier", "term_months", "bki_has_history", "bki_open_loans", "bki_closed_loans", "bki_overdue_30_cnt", "bki_overdue_60_cnt",
             "bki_overdue_90_cnt", "bki_max_overdue_days", "bki_defaults_cnt", "bki_inquiries_3m", "bki_inquiries_12m", "fns_account_blocked", "fns_mass_address",
             "fns_mass_director", "fns_disqualified", "fns_bankruptcy", "bank_tenure_months", "counterparties_cnt", "bki_history_months"}


def _py(col: str, v):
    if v is None or (isinstance(v, float) and math.isnan(v)) or v is pd.NA:
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        v = float(v)
        return int(v) if v.is_integer() and (col in _INT_COLS or col.rsplit("_q", 1)[0] in {c for _, c in FIN}) else v
    return v


def quarter_labels(app_date: str) -> list[str]:
    d = pd.Timestamp(app_date)
    y, q = d.year, (d.month - 1) // 3          # квартал даты заявки; последний полный = предыдущий
    labels = []
    for _ in range(8):
        q -= 1
        if q < 0:
            q, y = 3, y - 1
        labels.append(f"{y}-Q{q + 1}")
    return labels[::-1]


def month_labels(app_date: str) -> list[str]:
    d = pd.Timestamp(app_date).to_period("M")
    return [str(d - i) for i in range(12, 0, -1)]


def row_to_fixture(r: dict) -> dict:
    g = lambda c: _py(c, r.get(c))
    ql, ml = quarter_labels(r["application_date"]), month_labels(r["application_date"])
    fin = [dict(quarter=ql[k], **{a: g(f"{c}_q{k + 1}") for a, c in FIN}) for k in range(8)]
    months = [dict(month=ml[k], **{a: g(f"{c}{k + 1}") for a, c in TURN}) for k in range(12)]
    return dict(
        _synthetic=True,
        company={a: g(c) for a, c in COMPANY},
        financials=fin,
        accountTurnover=dict(**{a: g(c) for a, c in TURN_SCALAR}, months=months),
        creditHistory={a: g(c) for a, c in BKI},
        fns={a: g(c) for a, c in FNS},
        application={a: g(c) for a, c in APP},
    )


def fixture_to_row(f: dict) -> dict:
    nan = lambda v: np.nan if v is None else v
    r = {c: nan(f["company"][a]) for a, c in COMPANY}
    for k, q in enumerate(f["financials"]):
        r.update({f"{c}_q{k + 1}": nan(q[a]) for a, c in FIN})
    at = f["accountTurnover"]
    r.update({c: nan(at[a]) for a, c in TURN_SCALAR})
    for k, m in enumerate(at["months"]):
        r.update({f"{c}{k + 1}": nan(m[a]) for a, c in TURN})
    r.update({c: nan(f["creditHistory"][a]) for a, c in BKI})
    r.update({c: nan(f["fns"][a]) for a, c in FNS})
    r.update({c: nan(f["application"][a]) for a, c in APP})
    return r

Overwriting ../data/fixtures.py


## 1. Скоринг демо-пула (600 синтетических компаний с датой заявки 2026-10-26)

In [2]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "risk-scoring")); sys.path.insert(0, str(ROOT))
from riskscore.engine import ScoringEngine
from data.generate import apply_scenario, is_valid_inn, DATASET_VERSION
from data import fixtures as FX
STR = {"inn": str, "ogrn": str, "okved": str, "region_code": str}
pool = pd.read_csv(ROOT / "data/generated/demo_pool_raw.csv", dtype=STR).drop(columns=["default_flag", "oracle_pd"])
eng = ScoringEngine(ROOT / "risk-scoring/models")
TS = "2026-10-26T12:00:00Z"
GR = eng.grades

def reference_decision(res, requested):
    """ИЛЛЮСТРАТИВНАЯ политика credit-core (пороги живут в Backend-конфигурации, не в модели)."""
    crit = [f for f in res["policyFlags"] if f["severity"] == "CRITICAL"]
    warn = [f for f in res["policyFlags"] if f["severity"] == "WARNING"]
    lim, g = res["recommendedLimit"], res["ratingGrade"]
    if crit: return "REJECT_HARD_STOP"
    if g in ("C", "D"): return "REJECT_RISK"
    if lim == 0: return "REJECT_NO_CAPACITY"
    if warn or g in ("B", "CCC", "CC"): return "MANUAL_REVIEW"
    if lim >= requested: return "AUTO_APPROVE"
    if lim >= 0.3 * requested: return "COUNTER_OFFER"
    return "REJECT_NO_CAPACITY"

def info(df):
    rs = eng.score_frame(df, TS)
    return rs, pd.DataFrame({"pd": [r["pd"] for r in rs], "grade": [r["ratingGrade"] for r in rs], "limit": [r["recommendedLimit"] for r in rs],
                             "req": df.requested_amount.to_numpy(), "flg": [[f["code"] for f in r["policyFlags"]] for r in rs],
                             "crit": [any(f["severity"] == "CRITICAL" for f in r["policyFlags"]) for r in rs],
                             "dec": [reference_decision(r, q) for r, q in zip(rs, df.requested_amount)]})
res0, inf0 = info(pool)
print(inf0.dec.value_counts().to_string()); print("\nрейтинги в пуле:", inf0.grade.value_counts().reindex(GR).fillna(0).astype(int).to_dict())

dec
AUTO_APPROVE          185
REJECT_NO_CAPACITY    153
MANUAL_REVIEW         126
REJECT_RISK            58
COUNTER_OFFER          49
REJECT_HARD_STOP       29

рейтинги в пуле: {'AAA': 76, 'AA': 138, 'A': 117, 'BBB': 56, 'BB': 38, 'B': 33, 'CCC': 38, 'CC': 37, 'C': 50, 'D': 17}


## 2. Подбор 15 сценарных кейсов
- кейсы «по рейтингу» (автоодобрение, контр-оферта, серая зона, отказ по риску) выбираются из пула по **фактическому результату модели**;
- кейсы «по флагам политики» берутся из хороших заёмщиков (без флагов, рейтинг ≤ BBB) с **жёстким оверрайдом** одного фактора — так исход однозначно объясним.

In [3]:
used = set(); cases = []
def take(mask, key, scenario, desc, override=None):
    cand = inf0[mask & ~inf0.index.isin(used)].sort_values(key)
    assert len(cand), f"нет кандидата для {scenario}"
    i = int(cand.index[len(cand) // 2] if key == "mid" else cand.index[0]); used.add(i); cases.append(dict(idx=i, scenario=scenario, desc=desc, override=override)); return i
inf0["mid"] = 0
nofl = inf0.flg.map(len) == 0
take((inf0.dec == "AUTO_APPROVE") & inf0.grade.isin(["AAA", "AA", "A"]) & nofl & (inf0.limit >= 1.2 * inf0.req), "pd", "AUTO_APPROVE_PRIME", "Сильный заёмщик: рейтинг A и выше, флагов нет, лимит ≥ 120% запроса — автоодобрение")
take((inf0.dec == "AUTO_APPROVE") & inf0.grade.isin(["BBB", "BB"]) & nofl, "pd", "AUTO_APPROVE_STANDARD", "Типичный заёмщик (BBB/BB): лимит покрывает запрос, флагов нет")
take((inf0.dec == "COUNTER_OFFER") & nofl, "pd", "COUNTER_OFFER_LIMIT", "Рекомендованный лимит ниже запрошенной суммы — контр-оферта по сумме/сроку")
take((inf0.dec == "MANUAL_REVIEW") & inf0.grade.isin(["B", "CCC"]) & nofl, "pd", "MANUAL_REVIEW_GREY_ZONE", "Серая зона по рейтингу (B/CCC) без флагов → ручное рассмотрение андеррайтером")
take((inf0.dec == "REJECT_RISK") & ~inf0.crit & inf0.grade.isin(["C", "D"]), "pd", "REJECT_HIGH_RISK", "Высокий PD (рейтинг C/D) без стоп-флагов — отказ по риску")
good = (inf0.dec == "AUTO_APPROVE") & nofl & inf0.grade.isin(["AAA", "AA", "A", "BBB"])
for sc, code, desc in [("NO_HISTORY", "NO_CREDIT_HISTORY", "Нет кредитной истории (тонкое дело) — оценка менее надёжна, нужен ручной разбор"),
                       ("LIQUIDATION", "LIQUIDATION_IN_PROGRESS", "Компания в процессе ликвидации — hard-stop"),
                       ("ACCOUNT_BLOCKED", "ACCOUNT_BLOCKED", "Блокировка расчётных счетов — hard-stop"),
                       ("TAX_DEBT", "TAX_DEBT_SIGNIFICANT", "Значимый налоговый долг (6% годовой выручки) — требуется разбор"),
                       ("DELINQUENCY_90", "DELINQUENCY_90_PLUS", "Просрочка 90+ дней в кредитной истории — hard-stop"),
                       ("MASS_ADDRESS", "MASS_ADDRESS", "Массовый адрес регистрации — риск «технической» компании"),
                       ("DISQUALIFIED", "DISQUALIFIED_DIRECTOR", "Дисквалифицированный руководитель — hard-stop"),
                       ("YOUNG_BUSINESS", "YOUNG_BUSINESS", "Бизнесу ~4 месяца — недостаточно истории"),
                       ("BANKRUPTCY", "BANKRUPTCY_SIGNS", "Признаки банкротства — hard-stop"),
                       ("NEGATIVE_EQUITY", "NEGATIVE_EQUITY", "Отрицательный собственный капитал")]:
    take(good, "pd", f"FLAG_{code}", desc, override=(sc, code))
print(len(cases), "сценарных кейсов выбрано")

15 сценарных кейсов выбрано


## 3. Применяем оверрайды, добавляем 50 типичных компаний (по 5 на каждую рейтинговую градацию)

In [4]:
demo = pool.copy()
for c in cases:
    if c["override"]: apply_scenario(demo, c["idx"], c["override"][0])
res1, inf1 = info(demo)
for c in cases:
    if c["override"]:
        assert c["override"][1] in inf1.flg[c["idx"]], (c["scenario"], inf1.flg[c["idx"]])
rest = inf1[~inf1.index.isin(used)]
typical = []
for g in GR:
    typical += list(rest[rest.grade == g].sort_values("pd").index[:: max(1, len(rest[rest.grade == g]) // 5)][:5])
typical = [i for i in typical if i not in used][:50]
print("типичных компаний:", len(typical), "| по рейтингам:", inf1.loc[typical].grade.value_counts().reindex(GR).fillna(0).astype(int).to_dict())
order = [c["idx"] for c in cases] + typical
sel = demo.loc[order].copy(); sel_res = [res1[i] for i in order]; sel_inf = inf1.loc[order]
meta = {c["idx"]: (f"DC{k + 1:02d}", c["scenario"], c["desc"]) for k, c in enumerate(cases)}
for k, i in enumerate(typical): meta[i] = (f"T{k + 1:02d}", "TYPICAL", f"Типичная компания, рейтинг {inf1.grade[i]}")
print("всего демо-компаний:", len(sel), "(требование ТЗ ≥ 60; сценарных ≥ 12)")
print(sel_inf.loc[[c["idx"] for c in cases], ["grade", "pd", "limit", "req", "dec"]].assign(case=[c["scenario"] for c in cases]).to_string())

типичных компаний: 50 | по рейтингам: {'AAA': 5, 'AA': 5, 'A': 5, 'BBB': 5, 'BB': 5, 'B': 5, 'CCC': 5, 'CC': 5, 'C': 5, 'D': 5}
всего демо-компаний: 65 (требование ТЗ ≥ 60; сценарных ≥ 12)
    grade       pd     limit        req               dec                          case
276   AAA  0.00935    630000   220000.0      AUTO_APPROVE            AUTO_APPROVE_PRIME
478   BBB  0.04718    850000   630000.0      AUTO_APPROVE         AUTO_APPROVE_STANDARD
238   AAA  0.01195    130000   180000.0     COUNTER_OFFER           COUNTER_OFFER_LIMIT
443     B  0.12449    340000   390000.0     MANUAL_REVIEW       MANUAL_REVIEW_GREY_ZONE
313     C  0.28221         0   100000.0       REJECT_RISK              REJECT_HIGH_RISK
270   AAA  0.01210   5840000  1120000.0     MANUAL_REVIEW        FLAG_NO_CREDIT_HISTORY
494   AAA  0.01058  10550000  2150000.0  REJECT_HARD_STOP  FLAG_LIQUIDATION_IN_PROGRESS
221   AAA  0.01223   2250000   250000.0  REJECT_HARD_STOP          FLAG_ACCOUNT_BLOCKED
160   AAA  0.01172 

## 4. Запись фикстур: общий файл для сидирования БД и по одному файлу на источник для моков
Единые ИНН во всех файлах; в каждом файле `_synthetic: true` и текст-предупреждение.

In [5]:
FIX = ROOT / "data/fixtures"; FIX.mkdir(parents=True, exist_ok=True)
head = lambda n, desc: {"_synthetic": True, "_notice": FX.NOTICE, "_datasetVersion": DATASET_VERSION, "_asOf": "2026-10-26", "_description": desc, "_count": n}
full, by = [], {k: {} for k in ("egrul", "financials", "bki", "fns", "bank_turnover")}
for i, res in zip(order, sel_res):
    r = demo.loc[i].to_dict(); r["application_date"] = str(r["application_date"])
    f = FX.row_to_fixture(r); did, scen, desc = meta[i]
    dec = reference_decision(res, r["requested_amount"])
    f["demo"] = dict(caseId=did, scenario=scen, description=desc)
    f["expected"] = dict(referenceDecision=dec, policyFlags=[x["code"] for x in res["policyFlags"]], ratingGrade=res["ratingGrade"], riskSegment=res["riskSegment"],
                         pd=res["pd"], riskScore=res["riskScore"], recommendedLimit=res["recommendedLimit"], recommendedRate=res["recommendedRate"],
                         maxTermMonths=res["maxTermMonths"], modelVersion=res["modelVersion"],
                         note="referenceDecision - иллюстрация политики credit-core; модель отдаёт только оценку и флаги")
    full.append(f); inn = f["company"]["inn"]
    by["egrul"][inn] = f["company"]; by["financials"][inn] = f["financials"]; by["bki"][inn] = f["creditHistory"]
    by["fns"][inn] = f["fns"]; by["bank_turnover"][inn] = f["accountTurnover"]
json.dump({**head(len(full), "Полные синтетические карточки компаний + заявка + ожидаемый результат скоринга (сидирование БД, тесты)"), "companies": full},
          open(FIX / "companies.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
descr = dict(egrul="Сведения ЕГРЮЛ/ЕГРИП (профиль, статус)", financials="Финансовая отчётность за 8 кварталов", bki="Кредитная история (БКИ); hasHistory=0 -> остальные поля null",
             fns="Данные ФНС: задолженность, блокировки, массовый адрес/руководитель, дисквалификация, банкротство", bank_turnover="Обороты по расчётному счёту за 12 месяцев")
for k, v in by.items():
    json.dump({**head(len(v), descr[k] + " (ключ - ИНН)"), "byInn": v}, open(FIX / f"{k}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
sel_flat = sel.copy(); sel_flat.insert(0, "demo_case_id", [meta[i][0] for i in order]); sel_flat.insert(1, "scenario", [meta[i][1] for i in order])
sel_flat["synthetic_notice"] = "SYNTHETIC DATA"
sel_flat.to_csv(FIX / "companies_flat.csv", index=False, float_format="%.4f", lineterminator="\n")
print("файлы:", sorted(p.name for p in FIX.iterdir()), "| размер companies.json:", round((FIX / "companies.json").stat().st_size / 1e6, 2), "МБ")

файлы: ['bank_turnover.json', 'bki.json', 'companies.json', 'companies_flat.csv', 'demo_cases.csv', 'egrul.json', 'financials.json', 'fns.json'] | размер companies.json: 0.43 МБ


## 5. Таблица «ИНН → ожидаемое поведение» (`data/fixtures/demo_cases.csv`, `docs/test-data.md`)

In [6]:
rows = []
for i, res in zip(order, sel_res):
    r = demo.loc[i]
    rows.append(dict(caseId=meta[i][0], inn=r.inn, name=r.company_name, scenario=meta[i][1], product=r.product_code, requested=int(r.requested_amount),
                     termMonths=int(r.term_months), pd=res["pd"], ratingGrade=res["ratingGrade"], riskScore=res["riskScore"], recommendedLimit=res["recommendedLimit"],
                     recommendedRate=res["recommendedRate"], policyFlags=";".join(x["code"] for x in res["policyFlags"]),
                     referenceDecision=reference_decision(res, r.requested_amount), description=meta[i][2]))
tab = pd.DataFrame(rows); tab.to_csv(FIX / "demo_cases.csv", index=False, lineterminator="\n")
L = ["# Тестовые данные: демо-кейсы (ИНН → ожидаемое поведение)", "",
     "> **Все данные синтетические**: сгенерированы командой, к реальным лицам и организациям не относятся; ИНН/ОГРН имеют валидные контрольные суммы, но лежат в несуществующем диапазоне (регион `00`).", "",
     f"Версия датасета: `{DATASET_VERSION}`, seed 42, дата заявок демо-набора 2026-10-26, модель `{eng.version}`. Файлы: `data/fixtures/*.json` (по источнику и общий `companies.json`), `companies_flat.csv`, `demo_cases.csv`.", "",
     "**Как читать колонку «Референсное решение».** Это иллюстрация того, как `credit-core` может интерпретировать оценку (пороги — конфигурация Backend). Скоринг отдаёт `pd`, рейтинг, лимит и флаги; решение принимает не он. Иллюстративные правила: любой `CRITICAL`-флаг → отказ (hard-stop); рейтинг C/D или нулевой лимит → отказ; `WARNING`-флаг или рейтинг B/CCC/CC → ручное рассмотрение; лимит ≥ запроса → автоодобрение; лимит ≥ 30% запроса → контр-оферта.", "",
     "## Сценарные кейсы", "", "| Кейс | ИНН | Сценарий | Продукт | Запрос, ₽ | Срок | PD | Рейтинг | Лимит, ₽ | Флаги политики | Референсное решение |", "|---|---|---|---|---|---|---|---|---|---|---|"]
for r in rows[:len(cases)]:
    L.append(f"| {r['caseId']} | `{r['inn']}` | {r['scenario']}<br>_{r['description']}_ | {r['product']} | {r['requested']:,} | {r['termMonths']} | {r['pd']:.1%} | {r['ratingGrade']} | {r['recommendedLimit']:,} | {r['policyFlags'] or '—'} | **{r['referenceDecision']}** |".replace(",", " "))
L += ["", "## Типичные компании (по 5 на рейтинговую градацию)", "", "| Кейс | ИНН | Продукт | Запрос, ₽ | PD | Рейтинг | Лимит, ₽ | Флаги | Референсное решение |", "|---|---|---|---|---|---|---|---|---|"]
for r in rows[len(cases):]:
    L.append(f"| {r['caseId']} | `{r['inn']}` | {r['product']} | {r['requested']:,} | {r['pd']:.1%} | {r['ratingGrade']} | {r['recommendedLimit']:,} | {r['policyFlags'] or '—'} | {r['referenceDecision']} |".replace(",", " "))
(ROOT / "docs/test-data.md").write_text("\n".join(L) + "\n", encoding="utf-8")
print(tab.referenceDecision.value_counts().to_string())
print(tab[["caseId", "inn", "scenario", "ratingGrade", "recommendedLimit", "referenceDecision"]].head(15).to_string(index=False))

referenceDecision
REJECT_NO_CAPACITY    17
MANUAL_REVIEW         14
AUTO_APPROVE          12
REJECT_RISK            9
REJECT_HARD_STOP       9
COUNTER_OFFER          4
caseId          inn                     scenario ratingGrade  recommendedLimit referenceDecision
  DC01   0092935763           AUTO_APPROVE_PRIME         AAA            630000      AUTO_APPROVE
  DC02   0008932140        AUTO_APPROVE_STANDARD         BBB            850000      AUTO_APPROVE
  DC03 002899265408          COUNTER_OFFER_LIMIT         AAA            130000     COUNTER_OFFER
  DC04   0006160498      MANUAL_REVIEW_GREY_ZONE           B            340000     MANUAL_REVIEW
  DC05 002958657907             REJECT_HIGH_RISK           C                 0       REJECT_RISK
  DC06 002924606206       FLAG_NO_CREDIT_HISTORY         AAA           5840000     MANUAL_REVIEW
  DC07   0010199184 FLAG_LIQUIDATION_IN_PROGRESS         AAA          10550000  REJECT_HARD_STOP
  DC08   0088580317         FLAG_ACCOUNT_BLOCKED        

## 6. Проверки согласованности фикстур (единые ИНН, round-trip, регрессия результата)

In [7]:
data = json.load(open(FIX / "companies.json", encoding="utf-8"))["companies"]
inns = [c["company"]["inn"] for c in data]
print("компаний:", len(data), "| уникальных ИНН:", len(set(inns)), "| все ИНН валидны:", all(is_valid_inn(i) for i in inns))
for k in by: assert set(json.load(open(FIX / f"{k}.json", encoding="utf-8"))["byInn"]) == set(inns), k
print("во всех 5 файлах источников те же ИНН:", True)
print("маркировка синтетики во всех файлах:", all(json.load(open(p, encoding="utf-8")).get("_synthetic") for p in FIX.glob("*.json")))
ok = 0
for c in data:
    r = FX.fixture_to_row(c); r["application_date"] = c["application"]["applicationDate"]
    out = eng.score_one(r, TS)
    ok += (out["pd"] == c["expected"]["pd"] and out["ratingGrade"] == c["expected"]["ratingGrade"] and out["recommendedLimit"] == c["expected"]["recommendedLimit"]
           and [f["code"] for f in out["policyFlags"]] == c["expected"]["policyFlags"])
print(f"регрессия: результат скоринга из JSON-фикстуры совпал с ожидаемым у {ok} из {len(data)} компаний")
print("диапазон ИНН и дат:", min(inns), max(inns), "| дата заявки:", {c["application"]["applicationDate"] for c in data})

компаний: 65 | уникальных ИНН: 65 | все ИНН валидны: True
во всех 5 файлах источников те же ИНН: True
маркировка синтетики во всех файлах: True


регрессия: результат скоринга из JSON-фикстуры совпал с ожидаемым у 65 из 65 компаний
диапазон ИНН и дат: 0000696388 0098795826 | дата заявки: {'2026-10-26'}


In [8]:
%%writefile ../risk-scoring/tests/test_fixtures.py
"""Фикстуры: единые ИНН, маркировка синтетики, round-trip и регрессия ожидаемых результатов."""
import json
import sys
from pathlib import Path

import pytest

ROOT = Path(__file__).resolve().parents[2]
sys.path[:0] = [str(ROOT / "risk-scoring"), str(ROOT)]
from data import fixtures as FX                       # noqa: E402
from data.generate import is_valid_inn                # noqa: E402
from riskscore.engine import ScoringEngine            # noqa: E402

FIX = ROOT / "data/fixtures"
TS = "2026-10-26T12:00:00Z"


@pytest.fixture(scope="module")
def companies():
    return json.loads((FIX / "companies.json").read_text(encoding="utf-8"))["companies"]


def test_at_least_60_demo_companies_and_12_scenarios(companies):
    assert len(companies) >= 60
    assert len([c for c in companies if c["demo"]["scenario"] != "TYPICAL"]) >= 12


def test_all_fixture_files_marked_synthetic_and_share_same_inns(companies):
    inns = {c["company"]["inn"] for c in companies}
    assert len(inns) == len(companies) and all(is_valid_inn(i) and i.startswith("00") for i in inns)
    for p in FIX.glob("*.json"):
        d = json.loads(p.read_text(encoding="utf-8"))
        assert d["_synthetic"] is True and "СИНТЕТИЧЕСКИЕ" in d["_notice"]
        if "byInn" in d:
            assert set(d["byInn"]) == inns


def test_fixture_roundtrip_and_expected_results_reproduce(companies):
    eng = ScoringEngine(ROOT / "risk-scoring/models")
    for c in companies:
        row = FX.fixture_to_row(c)
        again = FX.row_to_fixture(row)
        assert again["company"] == c["company"] and again["financials"] == c["financials"] and again["creditHistory"] == c["creditHistory"]
        out = eng.score_one(row, TS)
        e = c["expected"]
        assert (out["pd"], out["ratingGrade"], out["recommendedLimit"]) == (e["pd"], e["ratingGrade"], e["recommendedLimit"])
        assert [f["code"] for f in out["policyFlags"]] == e["policyFlags"]


def test_no_credit_history_case_has_null_bki_fields(companies):
    c = next(c for c in companies if c["demo"]["scenario"] == "FLAG_NO_CREDIT_HISTORY")
    assert c["creditHistory"]["hasHistory"] == 0 and c["creditHistory"]["openLoans"] is None

Overwriting ../risk-scoring/tests/test_fixtures.py


In [9]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(ROOT / "risk-scoring/tests")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-1500:], r.stderr[-800:])

...........................                                              [100%]
=============================== warnings summary ===============================
risk-scoring/tests/test_scoring.py::test_missing_columns_and_garbage_do_not_crash
  /home/arakteni/alfa-credit-scoring/risk-scoring/tests/test_scoring.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'abc' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
    bad.loc[bad.index[1], "requested_amount"] = "abc"

risk-scoring/tests/test_scoring.py::test_empty_minimal_input_does_not_crash
  /home/arakteni/alfa-credit-scoring/risk-scoring/riskscore/features.py:229: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_si